In [ ]:
# Install necessary packages (run this if needed)
# !pip install torch torchvision pandas matplotlib seaborn scikit-learn pillow tqdm lightning

# Import libraries
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from PIL import Image
import lightning as L
from lightning.pytorch.loggers import CSVLogger

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
! curl -L -o skin-cancer-mnist-ham10000.zip https://www.kaggle.com/api/v1/datasets/download/kmader/skin-cancer-mnist-ham10000
! unzip skin-cancer-mnist-ham10000.zip -d data/

In [ ]:
metadata = pd.read_csv('data/HAM10000_metadata.csv')
class_counts = metadata['dx'].value_counts()
print("Class distribution:")
for class_name, count in class_counts.items():
    print(f"{class_name}: {count} images ({count/len(metadata)*100:.2f}%)")

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform
        self.classes = sorted(df['dx'].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_id = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.image_dir, 'data/HAM10000_images_part_1', f"{img_id}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.image_dir, 'data/HAM10000_images_part_2', f"{img_id}.jpg")
        image = Image.open(img_path).convert('RGB')
        diagnosis = self.df.iloc[idx]['dx']
        label = self.class_to_idx[diagnosis]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}


class SkinLesionDataModule(L.LightningDataModule):
    def __init__(
        self,
        metadata: pd.DataFrame,
        image_dir: str = '.',
        batch_size: int = 32,
        num_workers: int = 4,
        test_size: float = 0.2,
        random_state: int = 42,
    ):
        super().__init__()
        self.metadata = metadata
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.test_size = test_size
        self.random_state = random_state

        self.train_dataset = None
        self.val_dataset = None
        self.class_names = None

    def setup(self, stage=None):
        train_df, val_df = train_test_split(
            self.metadata,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=self.metadata['dx'],
        )

        self.train_dataset = SkinLesionDataset(
            df=train_df,
            image_dir=self.image_dir,
            transform=data_transforms['train'],
        )

        self.val_dataset = SkinLesionDataset(
            df=val_df,
            image_dir=self.image_dir,
            transform=data_transforms['val'],
        )

        self.class_names = self.train_dataset.classes

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )


# Initialize data module
dm = SkinLesionDataModule(
    metadata=metadata,
    image_dir='.',
    batch_size=32,
    num_workers=4,
)
dm.setup()

print(f"Training set size: {len(dm.train_dataset)}")
print(f"Validation set size: {len(dm.val_dataset)}")
class_names = dm.class_names
print(f"Class names: {class_names}")

In [ ]:
class ResNetTransferModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.resnet = torchvision.models.resnet50(weights="IMAGENET1K_V1")

        # Freeze the early layers
        for param in list(self.resnet.parameters())[:-20]:
            param.requires_grad = False

        # Replace the final fully connected layer
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.resnet(x)


class LitResNetTransferModel(L.LightningModule):
    def __init__(self, num_classes=7, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = ResNetTransferModel(num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=x.size(0))
        self.log("train_acc", acc, on_step=False, on_epoch=True, prog_bar=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=x.size(0))
        self.log("val_acc", acc, on_step=False, on_epoch=True, prog_bar=True, batch_size=x.size(0))

    def predict_step(self, batch, batch_idx):
        x, y = batch
        preds = torch.argmax(self(x), dim=1)
        return preds, y

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.hparams.lr)


# ── Train ──────────────────────────────────────────────────────────────────────
num_epochs_transfer = 10

lit_transfer_model = LitResNetTransferModel(num_classes=len(class_names), lr=1e-4)
logger = CSVLogger(save_dir=".", name="lightning_logs", version="resnet_transfer")

trainer = L.Trainer(
    max_epochs=num_epochs_transfer,
    accelerator="auto",
    logger=logger,
    log_every_n_steps=1,
)
trainer.fit(lit_transfer_model, datamodule=dm)

torch.save(lit_transfer_model.model.state_dict(), "skin_lesion_transfer_learning.pth")

# ── Evaluate ───────────────────────────────────────────────────────────────────
raw_preds = trainer.predict(lit_transfer_model, dataloaders=dm.val_dataloader())
y_pred_tl = torch.cat([p for p, _ in raw_preds]).cpu().numpy()
y_true_tl = torch.cat([y for _, y in raw_preds]).cpu().numpy()

In [ ]:
# ── Classification report ──────────────────────────────────────────────────────
print("Classification Report (Transfer Learning):")
print(classification_report(y_true_tl, y_pred_tl, target_names=class_names))

# ── Confusion matrix ───────────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
cm_tl = confusion_matrix(y_true_tl, y_pred_tl)
sns.heatmap(cm_tl, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Transfer Learning)")
plt.tight_layout()
plt.show()

# ── Learning curves from CSVLogger ────────────────────────────────────────────
metrics_path = Path(trainer.logger.log_dir) / "metrics.csv"
metrics = pd.read_csv(metrics_path)

learning_curves = (
    metrics.groupby("epoch", as_index=False)[["train_loss", "val_loss", "train_acc", "val_acc"]]
    .last()
    .sort_values("epoch")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

learning_curves.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], marker="o")
axes[0].set_title("Loss (Transfer Learning)")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.3)

learning_curves.plot(x="epoch", y=["train_acc", "val_acc"], ax=axes[1], marker="o")
axes[1].set_title("Accuracy (Transfer Learning)")
axes[1].set_ylabel("Accuracy")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

learning_curves.tail()